# Integrate datasets

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the per-source notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `other_sources.ipynb`)._

**Purpose:** this is where we combine the per-source explorations into one dataset once each notebook has settled on the fields it wants to keep. It stays a placeholder until that's ready — there's nothing to merge yet.

**Before filling this in:**
- Each source notebook should have decided what it exports per project/package (and under what column name — e.g. `repo_owner`/`repo_name`, or a full repo URL) so we have a consistent join key across sources.
- Merging happens here, not in the individual notebooks — keep source notebooks focused on exploring their own dataset.

In [ ]:
from google.cloud import bigquery
from decay_before_archival import get_client

client = get_client()

## Join keys per source

| Source | Join key column(s) | Notes |
|---|---|---|
| GH Archive (base table) | `repo_full_name` (`repo_id` is also stable) | 62,581 repos; every other source is left-joined onto this |
| deps.dev | `repo_full_name_lower` | matched on `LOWER(repo_full_name)`; only repos known to deps.dev appear |
| OpenSSF Scorecard | _TODO_ | |
| PyPI download stats | _TODO_ | |

In [ ]:
import pandas as pd

gh = pd.read_csv("../data/gh_archive_features_2024_clean.csv")
gh["repo_key"] = gh["repo_full_name"].str.lower()

deps = pd.read_csv("../data/deps_dev_features_2024.csv").rename(columns={"repo_full_name_lower": "repo_key"})
print(f"GH Archive: {len(gh):,} rows ({gh['repo_key'].nunique():,} unique keys) | deps.dev: {len(deps):,} rows ({deps['repo_key'].nunique():,} unique keys)")

**Notes:**

GH Archive contains 62,627 rows with 62,581 unqiue keys where deps.dev contains 1,405 rows with the same number of unique keys

## Merge (TODO)

Combine the sources on the join key(s) above (e.g. `pandas.merge`, or a BigQuery `JOIN` if it's cleaner to do it in SQL).

In [ ]:
# Left join: keep every GH Archive repo, add deps.dev columns where we have them.
merged = gh.merge(deps.assign(in_depsdev=1), on="repo_key", how="left", validate="m:1")

# Indicators: a repo missing from deps.dev is structurally missing, not randomly missing.
merged["in_depsdev"] = merged["in_depsdev"].fillna(0).astype(int)
merged["has_package"] = (merged["n_packages"].fillna(0) > 0).astype(int)

# Package/version counts are true zeros for repos with no package; stars/forks/issues stay NaN (unknown, not zero).
count_cols = ["n_packages", "n_ecosystems", "n_versions_total", "versions_2024", "versions_2023", "advisory_links"]
merged[count_cols] = merged[count_cols].fillna(0)
merged.shape

**Notes:**

_(add observations here)_

## Match rate (TODO)

Report how many rows matched across sources vs. how many were dropped, so we know how much coverage loss the join introduced.

In [ ]:
n = len(gh)
matched = int(merged["in_depsdev"].sum())
with_pkg = int(merged["has_package"].sum())
unmatched_deps = len(set(deps["repo_key"]) - set(gh["repo_key"]))  # deps.dev rows with no GH Archive repo (should be 0)

print(pd.DataFrame({
    "stage": ["GH Archive repos (base)", "matched in deps.dev", "lost (not in deps.dev)", "matched AND has a package", "deps.dev rows with no GH match"],
    "rows": [n, matched, n - matched, with_pkg, unmatched_deps],
    "pct_of_base": [100.0, 100 * matched / n, 100 * (n - matched) / n, 100 * with_pkg / n, None],
}).round(2).to_string(index=False))

# Missing values per column, overall and among matched rows only (separates join-caused gaps from real nulls).
m = merged.loc[merged["in_depsdev"] == 1]
missing = pd.DataFrame({
    "missing_all": merged.isna().sum(),
    "pct_all": (100 * merged.isna().mean()).round(2),
    "missing_matched_only": m.isna().sum(),
    "pct_matched_only": (100 * m.isna().mean()).round(2),
})
missing[missing["missing_all"] > 0].sort_values("missing_all", ascending=False)

In [ ]:
# Does matching correlate with the label? (inactive_next_365d rate by deps.dev status)
merged.groupby(["in_depsdev", "has_package"])["inactive_next_365d"].agg(rows="size", inactive_rate="mean").round(3)

In [ ]:
# Flip to True only once all sources are joined and the duplicate-name issue is settled.
WRITE_MERGED = False

if WRITE_MERGED:
    merged.to_csv("../data/merged_dataset.csv", index=False)
    print(f"wrote {len(merged):,} rows x {merged.shape[1]} cols")

**Notes:**

- **Match rate:** 1,405 of 62,627 GH Archive rows matched deps.dev (2.2%); 61,222 were lost (not in deps.dev). All 1,405 deps.dev rows match a GH Archive repo, so nothing was dropped on that side.
- **Inactive rate (`inactive_next_365d`) by deps.dev status:**

  | Group | Repos | Inactive rate |
  |---|---|---|
  | Not in deps.dev | 61,222 | 85.6% |
  | In deps.dev, no package | 454 | 63.9% |
  | Has a package | 951 | 44.9% |

  This is a correlation, not a cause: published packages tend to be more mature and popular (the popularity bias the proposal flags). The 85.6% base rate also says most of this sample is small and quiet — worth checking against how the GH Archive sample was drawn.
- **Missingness:** `stars_end` / `forks_end` / `open_issues_end` are missing for 165 matched rows (package-only repos); the `_start` columns for 854. These gaps come from the join, so use `in_depsdev` / `has_package` indicators rather than imputing. `days_since_last_push_at_cutoff` is the only GH Archive column with real gaps (1.9% overall).
- **Duplicate names:** GH Archive has 62,627 rows but 62,581 unique names — 46 repos share a name across different `repo_id`s, and the name join gives them identical deps.dev values. Decide whether to dedupe before the final merge.
- **TODO:** `in_projects` shows as missing for every unmatched repo; fill it with 0 in the merge cell. The merged file is intentionally not written yet (`WRITE_MERGED = False`) until the other sources are joined.